# Mapa de linhas

Notebook para criar um mapa contendo as regiões com maior número de ocorrências de roubos a ônibus, sobreposto pelas linhas com maior incidencia

In [1]:
#!pip install folium
import folium
import geopandas as gpd
from shapely import wkt
import pandas as pd
from folium.plugins import HeatMap
import pickle

Carregando o dataframe contendo o risco de cada cisp

In [2]:
with open("../datasets/dataframe.pkl","rb") as file:
    df = pickle.load(file)
df = df[["cisp","furto_coletivo","roubo_em_coletivo","ano"]]
df["occur_bus"] = df["furto_coletivo"] +df["roubo_em_coletivo"]
df = df[df["ano"]==2024]
df.head()
df = df.groupby("cisp")["occur_bus"].sum() / 12
df= df.reset_index()
df

,cisp,occur_bus
0,1,19.750000
1,4,26.916667
2,5,14.500000
3,6,35.000000
4,7,3.083333
5,9,23.000000
6,10,27.250000
7,11,1.333333
8,12,16.500000
9,13,6.000000


Carregando os dados de localização das cisps e criando o mapa, colorido pelos riscos

In [3]:
cisps = gpd.read_file('../datasets/cisp_geo_data/lm_cisp_bd.shp')
cisps

,cisp,aisp,shape_Leng,shape_Area,AREA_GEO,geometry
0,120,35,1.595433,0.082321,9.375473e+08,"POLYGON ((-42.18687 -22.55548, -42.18733 -22.5..."
1,127,25,0.697487,0.006180,7.027806e+07,"MULTIPOLYGON (((-41.90082 -22.78264, -41.90079..."
2,151,11,1.752923,0.081815,9.334145e+08,"POLYGON ((-42.52555 -22.16087, -42.5006 -22.18..."
3,107,38,1.209676,0.050835,5.805239e+08,"POLYGON ((-43.34665 -22.00375, -43.34603 -22.0..."
4,123,32,2.118860,0.106640,1.216847e+09,"MULTIPOLYGON (((-41.97224 -22.1428, -41.97201 ..."
...,...,...,...,...,...,...
132,76,12,0.338630,0.000828,9.410025e+06,"MULTIPOLYGON (((-43.11375 -22.8646, -43.11358 ..."
133,81,12,0.593683,0.005718,6.495636e+07,"MULTIPOLYGON (((-42.98891 -22.89112, -42.9887 ..."
134,9,2,0.204071,0.000603,6.852825e+06,"POLYGON ((-43.16851 -22.91435, -43.16854 -22.9..."
135,1,5,0.306591,0.000308,3.503511e+06,"MULTIPOLYGON (((-43.17829 -22.89257, -43.1795 ..."


In [4]:
m = folium.Map(location=[-22.9068, -43.1729], zoom_start=11, tiles='cartodbpositron')

In [5]:
folium.Choropleth(
    geo_data=cisps.to_json(),          
    name='Risco por CISP',
    data=df,                       
    columns=['cisp', 'occur_bus'],    
    key_on='feature.properties.cisp', 
    fill_color='YlOrRd',                 # Escala: Yellow (baixo) to Red (alto)
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Taxa de Violência Contra Ônibus'
).add_to(m)

aux = pd.read_csv("../datasets/shapes_geom.csv")
aux = aux.drop_duplicates(subset=["shape_id"])
aux 

,feed_version,feed_start_date,feed_end_date,shape_id,shape,shape_distance,start_pt,end_pt,versao_modelo
0,2023-12-16,2023-12-16,2023-12-20,1oge,"LINESTRING(-43.232036 -22.923777, -43.23146 -2...",24466.1,POINT(-43.232036 -22.923777),POINT(-43.36544 -23.0016),201d79faee763526a030ff998bebea9782efe961
1,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8
2,2025-11-08,2025-11-08,2025-11-18,35bu,"MULTILINESTRING((-43.17754 -22.90107, -43.1775...",33079.0,POINT(-43.17754 -22.90107),POINT(-43.39397 -22.95665),2a53b9c61f4b76fe07761a33083c714f9c35bcd0
3,2025-10-04,2025-10-04,2025-10-11,iz18,"LINESTRING(-43.19987 -22.93997, -43.19975 -22....",12591.1,POINT(-43.19987 -22.93997),POINT(-43.2238 -22.98097),40c2bc268bdc2e22bc2474d99bca38dcb9ea4c40
4,2024-03-11,2024-03-11,2024-03-17,2ibq,"LINESTRING(-43.621884 -22.968783, -43.62189 -2...",24430.2,POINT(-43.621884 -22.968783),POINT(-43.4637 -22.87668),201d79faee763526a030ff998bebea9782efe961
...,...,...,...,...,...,...,...,...,...
13776,2023-03-01,2023-03-01,2023-03-15,krez,"MULTILINESTRING((-43.36332 -22.80502, -43.3638...",28149.7,POINT(-43.36332 -22.80502),POINT(-43.2753 -22.90216),201d79faee763526a030ff998bebea9782efe961
13868,2023-03-01,2023-03-01,2023-03-15,n6x6,"LINESTRING(-43.18086 -22.90192, -43.1791 -22.9...",23999.8,POINT(-43.18086 -22.90192),POINT(-43.342379 -22.872688),201d79faee763526a030ff998bebea9782efe961
13877,2026-03-14,2026-03-14,2026-03-27,gzrt,"MULTILINESTRING((-43.22883 -22.88588, -43.2281...",18883.0,POINT(-43.22883 -22.88588),POINT(-43.22304 -22.98473),3a6c7f1b4d7d9c5cc07568b119f9e07d65ab5cb7
13927,2023-06-01,2023-06-01,2023-06-15,u9o7_0,"LINESTRING(-43.18829 -22.91698, -43.18825 -22....",5900.6,POINT(-43.18829 -22.91698),POINT(-43.18422 -22.90219),201d79faee763526a030ff998bebea9782efe961


Carregando as linhas de ônibus e seus riscos, pegando as piores

In [6]:
with open("../datasets/rotas_riscos.pkl","rb") as file:
    linhas_riscos = pickle.load(file)
len(linhas_riscos)

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/rotas_riscos.pkl'

In [ ]:
l = pd.merge(linhas_riscos, aux, on="shape_id", how="left")
len(l)

612

In [ ]:
l

,route_id,shape_id,distancia_por_cisp,route_short_name,route_long_name,score,feed_version,feed_start_date,feed_end_date,shape,shape_distance,start_pt,end_pt,versao_modelo
0,E2336AAA0A,ab43,"[(40, 1.52), (5, 2.06), (27, 1.91), (1, 0.17),...",2336,Campo Grande - Castelo,30.345283,2025-07-16,2025-07-16,2025-10-03,"MULTILINESTRING((-43.17488 -22.90534, -43.1748...",54209.5,POINT(-43.17488 -22.90534),POINT(-43.55818 -22.90166),117a336a322996dccaeda55ed7b3f9061400c0ac
1,O0629AAA0A,ue02,"[(21, 2.13), (25, 4.88), (27, 5.54), (38, 1.08...",629,Irajá - Saens Peña,30.574410,2024-07-22,2024-07-22,2024-07-26,"LINESTRING(-43.23253 -22.92286, -43.23277 -22....",21357.2,POINT(-43.23253 -22.92286),POINT(-43.33267 -22.82484),f958f57d6b63df60eacf6155d8e385295d94fc65
2,O0774AAA0A,i9sp,"[(38, 7.94), (27, 4.65), (29, 4.88), (27, 4.64...",774,Madureira - Jardim América,27.337874,2024-03-11,2024-03-11,2024-03-17,"MULTILINESTRING((-43.3419 -22.87037, -43.34213...",17495.5,POINT(-43.3419 -22.87037),POINT(-43.32736 -22.80729),201d79faee763526a030ff998bebea9782efe961
3,O0112AAA0A,O0112AAA0AVDU03,"[(6, 5.81), (18, 0.05), (15, 4.91), (4, 1.31),...",112,Terminal Gentileza - Alto Gávea,23.353323,2023-11-01,2023-11-01,2023-11-30,"LINESTRING(-43.23775 -22.98213, -43.23763 -22....",18001.0,POINT(-43.23775 -22.98213),POINT(-43.20869 -22.90045),201d79faee763526a030ff998bebea9782efe961
4,O0343AAA0A,j7v9,"[(18, 0.05), (26, 2.32), (1, 1.8), (4, 1.31), ...",343,Jardim Oceânico - Candelária,37.796880,2025-07-16,2025-07-16,2025-10-03,"MULTILINESTRING((-43.18188 -22.90225, -43.1818...",40080.0,POINT(-43.18188 -22.90225),POINT(-43.31259 -23.00551),117a336a322996dccaeda55ed7b3f9061400c0ac
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607,SE010,uijs,"[(41, 0.84), (32, 7.9), (28, 3.04), (30, 0.77)...",SE010,Terminal Paulo da Portela - Estação Morro do O...,28.096453,2024-09-13,2024-09-13,2024-09-28,"MULTILINESTRING((-43.336781 -22.877618, -43.33...",13743.6,POINT(-43.336781 -22.877618),POINT(-43.40043 -22.97035),5d6ba3193f5d93ba8aa3bb4277ef03ccaac50665
608,O0862AAN0A,O0862AAA0AIDU01,"[(32, 0.9), (16, 14.28)]",SN862,Rio das Pedras - Barra da Tijuca,50.364295,2023-04-01,2023-04-01,2023-04-30,"MULTILINESTRING((-43.33563 -22.97289, -43.3351...",15190.1,POINT(-43.33563 -22.97289),POINT(-43.35747 -22.99942),201d79faee763526a030ff998bebea9782efe961
609,O0393AAE0A,58du,"[(1, 1.12), (39, 2.3), (17, 4.14), (4, 0.87), ...",SE393,Bangu - Enseada de Botafogo,29.484030,2025-12-27,2025-12-27,2026-01-02,"MULTILINESTRING((-43.177892 -22.900515, -43.17...",46667.0,POINT(-43.177892 -22.900515),POINT(-43.48696 -22.8974),ed92813e65f25bd40abac429ec41774aaf20a12d
610,O0238AAE0A,i40u,"[(7, 0.36), (9, 3.3), (12, 1.47), (19, 1.29), ...",SE238,Água Santa - Enseada de Botafogo,24.534192,2025-12-21,2025-12-21,2025-12-26,"MULTILINESTRING((-43.31321 -22.89894, -43.3134...",30526.0,POINT(-43.31321 -22.89894),POINT(-43.181698 -22.946124),22d4617d5bb2889f600112473f926c3a35f27ed9


In [ ]:
def carregar_wkt(texto):
    try:
        return wkt.loads(texto)
    except:
        return None
l['geometria'] = l['shape'].apply(carregar_wkt)
l = gpd.GeoDataFrame(l, geometry='geometria', crs="EPSG:4326")

In [ ]:

top_10_risco = l.sort_values(by='score') 
folium.GeoJson(
    top_10_risco.to_crs(epsg=4326),
    name='Rotas de Ônibus',
    style_function=lambda x: {
        'color': 'red',
        'weight': 3,
        'opacity': 0.8
    },
    tooltip=folium.GeoJsonTooltip(fields=['route_short_name','score'])
).add_to(m)

folium.LayerControl().add_to(m)

In [ ]:
m.save('mapa_test.html')